# Clinical, Demographic, Treatment, and Outcome Variables for Breast Cancer Patients Undergoing Radiotherapy Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.10en-qfg1/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.10en-qfg1/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print high-level dataset summary
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")
print(f"Published: {meta.datePublished}")
print(f"Identifier: {meta.identifier}")
print(f"Keywords: {meta.keywords}")
print(f"License: {meta.license}")


## 2. Data Overview
Review available record sets (tables), fields (columns), and their `@id`s.

- We will enumerate each record set and summarize the fields within each.

In [ ]:
# List all record sets by @id in the Croissant schema
record_sets = dataset.metadata.recordSet
if not record_sets:
    print('No record sets found in the dataset metadata.')
else:
    for rs in record_sets:
        print(f"Record set name: {rs.name if hasattr(rs, 'name') else '-'} @id: {rs['@id']}")
        if hasattr(rs, 'field'):
            fields = rs.field
            if fields:
                print("  Fields:")
                for fld in fields:
                    print(f"    - {fld.name if hasattr(fld, 'name') else '-'} (@id: {fld['@id']})")
        print()
# Store all record set @ids as a list for access in later steps
record_set_ids = [rs['@id'] if isinstance(rs, dict) else rs.__dict__.get('@id') for rs in record_sets] if record_sets else []
print(f"Found record sets: {record_set_ids}")

## 3. Data Extraction
Load data from each available record set into a Pandas DataFrame for analysis.

- Each record set and field is referenced by its `@id`.

In [ ]:
# Extract record set dataframes
dfs = dict()
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dfs[record_set_id] = df
        print(f'Loaded {len(df)} records for record set {record_set_id}')
    else:
        print(f'Record set {record_set_id} yielded no records.')
# If data present, print sample columns and rows of first record set
if dfs:
    main_record_set_id = list(dfs.keys())[0]
    print(f"Columns in record set {main_record_set_id}:\n{dfs[main_record_set_id].columns.tolist()}")
    display(dfs[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's process a numeric field for analysis.

- We select a numeric field (referenced by its `@id`).
- Filter records exceeding a threshold, normalize, and group by another field.

In [ ]:
# Define which record set and fields to use (@id-based)
# (Replace the IDs below to match your data from the overview above)
if dfs:
    record_set_id = main_record_set_id
    df = dfs[record_set_id]

    # For illustration, we guess likely candidate columns based on typical dataset structure
    numeric_field_candidates = [col for col in df.columns if any(x in col.lower() for x in ['age', 'bmi', 'weight', 'height', 'dose']) or pd.api.types.is_numeric_dtype(df[col])]
    group_field_candidates = [col for col in df.columns if any(x in col.lower() for x in ['sex', 'gender', 'diabetes', 'group', 'education', 'economic', 'surgery'])]
    print(f"Candidate numeric fields: {numeric_field_candidates}")
    print(f"Candidate group fields: {group_field_candidates}")
    
    # Choose an example field (update by actual field '@id' if known)
    numeric_field = numeric_field_candidates[0] if numeric_field_candidates else None
    group_field = group_field_candidates[0] if group_field_candidates else None
    print(f"Selected numeric_field: {numeric_field} (use the @id if available)")
    print(f"Selected group_field: {group_field} (use the @id if available)")

    if numeric_field is not None:
        threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f} : {len(filtered_df)} rows")
        display(filtered_df.head())

        # Normalize
        filtered_df = filtered_df.copy()
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"Normalized {numeric_field}:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by categorical field
        if group_field and group_field in df.columns:
            grouped = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped by {group_field} (mean {numeric_field}):")
            display(grouped.head())
else:
    print('No data frames found to perform EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib or pandas-native plotting.

In [ ]:
import matplotlib.pyplot as plt

# Example: Histogram and boxplot for the selected numeric field, colored by group field
if dfs and numeric_field is not None:
    plt.figure(figsize=(10, 5))
    df[numeric_field].hist(bins=15, color='lightblue', edgecolor='k')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.title(f'Histogram of {numeric_field}')
    plt.show()

    # Boxplot if a group field is available
    if group_field and group_field in df.columns:
        plt.figure(figsize=(10, 5))
        df.boxplot(column=numeric_field, by=group_field, grid=False)
        plt.title(f'{numeric_field} by {group_field}')
        plt.suptitle("")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
In this notebook, we:
- Used the `mlcroissant` library to load FAIR² dataset metadata and records from the official Croissant schema.
- Explored the available record sets and fields using their `@id` references.
- Extracted tabular data into Pandas DataFrames, filtered, normalized, and grouped by categorical fields.
- Visualized field distributions with histograms and boxplots.

This workflow can be adapted to any Croissant-compatible dataset for transparent, reproducible, and interoperable data science.